<a href="https://colab.research.google.com/github/MrSimple07/Advanced-Machine-Learning-ITMO/blob/main/Beeline_Interview_08_04_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Рекомендательные системы (Hard Skills)

Музыкальные рекомендации специфичны тем, что здесь важна последовательность и контекст (время суток, настроение).

- Алгоритмы: Подготовьтесь объяснить разницу между Collaborative Filtering (ALS, Matrix Factorization) и Content-based подходами.

- Deep Learning & Embeddings: Освежите знания по Word2Vec/Item2Vec для создания эмбеддингов треков. Будьте готовы обсудить архитектуры Two-Tower (для быстрого поиска кандидатов) и использование RNN/Transformers для учета последовательности прослушиваний.

- Метрики: Вас точно спросят про NDCG, MAP и MRR. Почему в музыке Recall@K важнее, чем Precision? Как бороться с "пузырем фильтров" (diversity vs accuracy)?


# 2. Системная архитектура и MLOps

Вакансия делает упор на FastAPI, Docker, Kubernetes и высоконагруженные системы.

- Двухуровневая схема: Будьте готовы нарисовать схему:

  1. Candidate Retrieval (отбор ~1000 кандидатов из миллионов с помощью FAISS/Hnswlib).

  2. Ranking (ранжирование отобранных кандидатов тяжелым Gradient Boosting или нейросетью).

- Инструменты: Как интегрировать модель в FastAPI? Как организовать мониторинг (Prometheus/Grafana) и логирование? Как Spark помогает готовить фичи для обучения?

# 3. Математический фундамент
Для крупных компаний это стандартный этап.

Линейная алгебра: SVD-разложение, косинусное сходство (Cosine Similarity).

Тервер и Статистика: Основы A/B тестирования. Как рассчитать размер выборки? Что такое p-value и доверительный интервал? Как оценивать статистическую значимость в условиях сетевых эффектов?

# 2. Метрики

В рекомендациях нам не просто важно угадать, а важно поставить угаданное на первые места.

- MRR (Mean Reciprocal Rank): На каком месте оказался первый полезный трек? Если на 1-м — это 1 балл, если на 2-м — 0.5 балла, на 10-м — 0.1. Идеально для поиска конкретной песни.

- MAP (Mean Average Precision): Учитывает, насколько "плотно" полезные треки идут в топе. Если из 10 песен 3 подошли, лучше, чтобы они были на 1, 2 и 3 местах, а не на 8, 9 и 10.

- NDCG (Normalized Discounted Cumulative Gain): Самая продвинутая метрика. Она учитывает "вес" позиции (чем выше, тем важнее) и то, что оценки могут быть не только "да/нет", а, например, от 1 до 5 звезд.

- Почему Recall@K важнее Precision в музыке?
Precision — это "насколько точно мы угадали из предложенного". Recall — "какую долю из всего, что могло понравиться, мы нашли".
В музыке миллионы треков. Если мы предложим 10 песен и пользователю понравится 2 — это нормально (Precision низкий). Но если мы вообще не смогли найти те 2 песни, которые он бы полюбил из всего каталога — это провал (Recall низкий). Пользователь не расстроится из-за пары "мимо", но он расстроится, если не найдет "ту самую музыку".

### 1. Оценка моделей: Precision, Recall, NDCG, MAP

В рекомендациях (RecSys) мы не просто классифицируем объект (да/нет), мы ранжируем список.
- Precision@K (Точность): Какая доля из $K$ предложенных нами треков действительно понравилась пользователю? (Важно, если мы хотим показать лишь топ-5 треков).

- Recall@K (Полнота): Какую долю из всего интересного пользователю мы смогли найти в нашем топе $K$? (Важно, чтобы не упустить "ту самую" песню).

  - Почему в музыке Recall важнее Precision? Потому что музыка — контент "длинного хвоста". Пользователь готов пропустить (skip) 3-4 песни, если 5-я окажется идеальной. Нам важнее найти эту жемчужину в океане треков, чем идеально угадать каждую позицию.

- MAP (Mean Average Precision): Усредненная точность для всех пользователей. Штрафует за то, что релевантные треки находятся низко в списке.

- NDCG (Normalized Discounted Cumulative Gain): «Золотой стандарт».

  - Discounted: Чем ниже в списке трек, тем меньше он "стоит" (потому что до него редко доскролливают).

  - Normalized: Приводим результат к диапазону от 0 до 1, чтобы сравнивать разных пользователей.
  
  - Gain: Считает пользу, даже если трек "немного" нравится (например, 3 звезды из 5), а не только "понравилось/не понравилось".

# 3. Современные архитектуры: Two-Tower и Sequence models

Когда у нас миллионы пользователей и песен, мы не можем прогонять "тяжелую" нейросеть для каждого сочетания.

### Two-Tower (Две башни)
Это стандарт индустрии для отбора кандидатов (Retrieval).

  1. Левая башня: Нейросеть превращает данные о пользователе (история, возраст, устройство) в вектор (числа).

  2. Правая башня: Нейросеть превращает данные о треке (жанр, аудио-фичи) в такой же по размеру вектор.

  - Магия: Векторы обучаются так, чтобы у похожих пар "пользователь-трек" они были направлены в одну сторону. В продакшене мы просто ищем ближайшие векторы треков к вектору пользователя через быстрый поиск (например, библиотеку FAISS).

### RNN и Transformers (Учет последовательности)
Обычный Collaborative Filtering не знает, в каком порядке вы слушали музыку. Но если вы слушали 3 колыбельные, 4-й песней вряд ли должен быть хеви-метал.

- RNN (LSTM/GRU): "Запоминают" состояние пользователя. Каждая новая песня обновляет "настроение" вектора пользователя.

- Transformers: Анализируют всю сессию целиком. Они понимают, что песня, прослушанная 5 минут назад, важнее той, что была год назад. Это позволяет делать рекомендации "в моменте".

# 4. Построение рекомендательных систем

### 1. Этап 1: Candidate Generation (Retrieval)
На этом этапе мы сокращаем пространство поиска с 50 миллионов треков до 500–1000 «кандидатов». Скорость здесь критична (latency < 50-100 мс).

- Matrix Factorization (ALS): Мы разлагаем матрицу взаимодействий на два набора векторов. Чтобы найти кандидатов, мы используем Approximate Nearest Neighbors (ANN), например, через алгоритм HNSW (Hierarchical Navigable Small World) или библиотеку FAISS.

- Two-Tower Architecture: Это стандарт для современных систем. Одна «башня» кодирует пользователя, вторая — трек. В проде мы храним эмбеддинги треков в векторной базе данных (например, Milvus, Pinecone или FAISS). На запрос пользователя мы мгновенно находим 500 ближайших соседей.


### 2. Этап 2: Ranking (Scoring)
Теперь у нас есть 500 кандидатов. Нам нужно отсортировать их по вероятности того, что пользователь нажмет "Play" или дослушает до конца. Здесь мы можем позволить себе использовать более тяжелые модели.

- Gradient Boosting (CatBoost, XGBoost, LightGBM): Это лидеры в ранжировании. Мы подаем на вход:

  - User features: Пол, возраст, предпочтения (вектор), время суток, история последних прослушиваний.

  - Item features: Жанр, длительность, популярность, теги.

  - Cross features: Разница между временем последней активности и текущим временем.

- Deep Learning (Wide & Deep, DeepFM): Модели, которые одновременно учитывают как простые признаки (Wide), так и сложные скрытые зависимости (Deep).

### 3. Этап 3: Re-ranking (Post-processing)
Финальный фильтр, где бизнес-логика и UX важнее математики:

- Diversity: Не давать 10 песен подряд одного артиста.

- Business rules: "Подмешивание" новинок (эти треки мы специально повышаем в выдаче, чтобы понять их качество).

- Filtering: Удаление треков, которые пользователь уже слушал (или наоборот, добавление их для ностальгии).

### 4. Инструментарий и Инфраструктура (MLOps)
Это то, что прописано в требованиях вашей вакансии:

- Offline Training: Обучение моделей на данных из Data Lake (S3/HDFS) с использованием Apache Spark.

- Feature Store: Единое место для хранения "фичей" (например, среднее количество прослушиваний пользователя за неделю). Часто реализуется через Redis для быстрого доступа.

- Online Serving: FastAPI принимает запрос, идет в Feature Store за данными пользователя, затем делает запрос к векторной базе (для кандидатов) и отправляет их в модель (ранжирование).

- Airflow: Оркестратор. Следит, чтобы пайплайн обучения переобучал модель каждую ночь.

- A/B тестирование:

  - Bucketing: Пользователи делятся на группы на основе ID.

  - Metrics: Мы смотрим не только на CTR (клики), но и на Retention (вернулся ли пользователь завтра) и Listening Time.

# A/B Testing:

A/B тестирование: как это работает в ML

  - Split: Пользователи делятся на группы по user_id (обычно через хеширование, чтобы один человек всегда попадал в одну группу).

  - Hypothesis: «Если мы внедрим нейросеть для ранжирования, время прослушивания (Listening Time) вырастет на 5%».

Metrics: Мы следим за:

  - North Star Metric: Время прослушивания (основная цель).

  - Guardrail Metrics: Метрики безопасности (например, чтобы количество «скипов» не выросло, а сервис не упал от нагрузки).

Significance: Проверка через t-тест или критерий Манна-Уитни. Важно: если у вас сотни миллионов прослушиваний, даже минимальные изменения будут "статистически значимыми", поэтому важно оценивать практическую значимость.